# Previsão de Preços — Ações e Criptomoedas (Portfólio GitHub)

Previsão de preços diários de **ações, ETFs e criptomoedas** com pipeline completo em fases. Fontes de histórico diário dos últimos 8 anos: **Yahoo Finance (principal)** e **Stooq (complemento)**.

**Objectivo da análise:** prever o preço de fecho do dia seguinte, comparar modelos clássicos vs. deep learning entre tipos de activo e validar com backtesting vs. buy & hold.

**Fases:** Bronze (ingestão) → Silver (limpeza) → Gold (features) → Split temporal (treino até 2021, validação 2022–2023, teste 2024+) → Modelos (Regressão Linear, Random Forest, XGBoost, LSTM) → Avaliação (RMSE, MAE, MAPE + backtesting) → 7 perguntas de negócio.

## Tickers (12)

| Tipo | Tickers |
|------|---------|
| Ações tech | TSLA, GOOGL, AAPL, MSFT, AMZN, NVDA, META |
| ETFs | IAU (ouro), SPY (S&P 500), QQQ (Nasdaq 100) |
| Criptomoedas | BTC-USD, ETH-USD |

## Estrutura da Fase 1 — Ingestão Bronze

STEP 1 (Setup) → STEP 2 (Extração dos 12 tickers) → STEP 3 (Escrita `bronze_market_data`) → STEP 4 (Schema) → STEP 5 (Amostra de todos os tickers)

In [0]:
%python
# Instala a yfinance (fonte principal) — %pip tem de ser a 1ª linha da célula
%pip install yfinance -q

In [0]:
%python
# ==============================================================================
# STEP 1: SETUP — SCHEMA DO PROJECTO
# ==============================================================================
# Objectivo: Criar a infraestrutura (schema) que todas as fases vão usar.
#
# Input:  —
# Output: workspace.market_project (schema)

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.market_project")
print("✅ Schema workspace.market_project pronto")

In [0]:
%python
# ==============================================================================
# STEP 2: EXTRAÇÃO — COTAÇÕES DIÁRIAS DOS 12 TICKERS (8 ANOS)
# ==============================================================================
# Objectivo: Extrair o histórico diário (Open, High, Low, Close, Volume) dos
# últimos 8 anos via Yahoo Finance, com Stooq como fonte complementar.
#
# Input:  Yahoo Finance (principal) + Stooq (complemento, equity)
# Output: DataFrame pandas `pdf` com os 12 tickers
#
# Operações:
#   - Funções de extração com fallback Yahoo → Stooq
#   - Validação mínima de histórico (>200 sessões)
#   - Resumo por ticker (linhas, data mín./máx., fonte)

import yfinance as yf
import requests, io, time
import pandas as pd
from datetime import datetime, timedelta

TICKERS = ["TSLA", "GOOGL", "AAPL", "MSFT", "AMZN", "NVDA", "META",
           "IAU", "SPY", "QQQ", "BTC-USD", "ETH-USD"]
# Janela: início FIXO em 2018-11-01 (estável entre execuções) e fim em HOJE+1
# (o parâmetro end do yfinance é EXCLUSIVO — +1 dia garante que a sessão de
# HOJE entra sempre na extracção, fim-de-semana incluído para cripto)
START_DATE = "2018-11-01"
END_DATE = (datetime.now() + timedelta(days=1)).strftime("%Y-%m-%d")
print(f"Janela de extracção: {START_DATE} → {END_DATE} (hoje incluído)")
COLS = ["date", "open", "high", "low", "close", "volume"]


def extrair_yfinance(ticker):
    df = (yf.Ticker(ticker)
          .history(start=START_DATE, end=END_DATE, interval="1d", auto_adjust=True)
          .reset_index()[["Date", "Open", "High", "Low", "Close", "Volume"]])
    df.columns = COLS
    return df


def extrair_stooq(ticker):
    simbolo = (ticker.replace("-USD", "") + ".us").lower()  # complemento equity
    url = (f"https://stooq.com/q/d/l/?s={simbolo}"
           f"&d1={START_DATE.replace('-', '')}&d2={END_DATE.replace('-', '')}&i=d")
    df = pd.read_csv(io.StringIO(requests.get(url, timeout=30).text))
    if len(df.columns) < 6:
        raise ValueError("sem dados no Stooq")
    df = df[["Date", "Open", "High", "Low", "Close", "Volume"]]
    df.columns = COLS
    return df


def extrair_serie(ticker):
    for nome, fn in [("Yahoo", extrair_yfinance), ("Stooq", extrair_stooq)]:
        try:
            df = fn(ticker)
            if len(df) > 200:  # valida mínimo de histórico
                return df, nome
            print(f"[{ticker}] {nome}: histórico insuficiente ({len(df)} linhas)")
        except Exception as e:
            print(f"[{ticker}] {nome} falhou: {e}")
        time.sleep(1)
    raise RuntimeError(f"Todas as fontes falharam para {ticker}")


frames = []
for t in TICKERS:
    df, fonte = extrair_serie(t)
    df["ticker"], df["fonte"] = t, fonte
    frames.append(df)
    print(f"[{t}] {len(df):>5} linhas | {df['date'].min()} → {df['date'].max()} | fonte: {fonte}")

pdf = pd.concat(frames, ignore_index=True)
# Normaliza datas mistas (tz-aware/naive) para data de calendário com utc=True
dt = pd.to_datetime(pdf["date"], utc=True)
pdf["date"] = dt.dt.tz_localize(None).dt.date
pdf["ingested_at"] = pd.Timestamp.now(tz="UTC")
print(f"\nTotal: {len(pdf):,} linhas | {pdf['ticker'].nunique()} tickers")

In [0]:
%python
# ==============================================================================
# STEP 3: ESCRITA — DELTA BRONZE PARTITIONADA
# ==============================================================================
# Objectivo: Escrever os 12 tickers na Delta bronze_market_data, particionada
# por ticker, ano e mês (evita milhares de ficheiros minúsculos por data única).
#
# Input:  pdf (STEP 2)
# Output: workspace.market_project.bronze_market_data
#
# Operações:
#   - Conversão para Spark DataFrame + colunas ano/mes
#   - Escrita Delta partitionBy(ticker, ano, mes)
#   - Verificação de linhas escritas e resumo por ticker

from pyspark.sql import functions as F

TABELA = "workspace.market_project.bronze_market_data"

sdf = (spark.createDataFrame(pdf)
       .withColumn("ano", F.year("date"))
       .withColumn("mes", F.month("date")))

(sdf.write.format("delta")
     .mode("overwrite")
     .partitionBy("ticker", "ano", "mes")
     .saveAsTable(TABELA))

print(f"✅ Tabela: {TABELA}")
print(f"Linhas escritas: {sdf.count():,} | Tickers: {sdf.select('ticker').distinct().count()}")
display(spark.table(TABELA).groupBy("ticker", "fonte").count().orderBy("ticker"))

In [0]:
-- ==============================================================================
-- STEP 4: VALIDAÇÃO — SCHEMA DA BRONZE
-- ==============================================================================
-- Objectivo: Ver o schema da tabela bronze_market_data antes da aprovação.
DESCRIBE TABLE workspace.market_project.bronze_market_data

In [0]:
-- ==============================================================================
-- STEP 5: VALIDAÇÃO — AMOSTRA DE TODOS OS TICKERS (96 LINHAS)
-- ==============================================================================
-- Objectivo: Mostrar as primeiras 8 sessões de cada um dos 12 tickers, com
-- colunas em português e volume em dólares (volume × fecho).
WITH amostra AS (
  SELECT ticker, date, open, high, low, close, volume,
         ROW_NUMBER() OVER (PARTITION BY ticker ORDER BY date) AS rn
  FROM workspace.market_project.bronze_market_data
)
SELECT ticker, date AS `Data`, open AS `Abrir`, high AS `Alto`, low AS `Baixo`,
       close AS `Fechar`, volume AS `Volume`,
       ROUND(volume * close, 0) AS `Volume (USD)`
FROM amostra
WHERE rn <= 8
ORDER BY ticker, `Data`